In [1]:
from openai import OpenAI

In [2]:
openai_client = OpenAI()

In [3]:
def llm(prompt: str) -> str:
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=[{"role": "user", "content": prompt}]
    )
    return response.output_text

In [4]:
question = "I just discovered the course. Can I join?"

In [5]:
data = llm(prompt=question)

In [6]:
print(data)

Absolutely — you can join if enrollment is still open.

A good next step is to:
1. Check whether there’s a deadline or capacity limit.
2. Review the course requirements or prerequisites.
3. Contact the course organizer/instructor if you’re unsure.

If you want, I can help you draft a quick message asking to join.


In [9]:
context = """
Question: I just discovered the course. Can I join?
Answer: Yes, you can join the course as it starts on 8th June 2026!
"""

In [10]:
prompt_template = """You are a course FAQ assistant, your role is to answer student queries
You will be asked a question, and you should answer it based on the following context:

{context}

Question: {question}
"""

In [11]:
prompt = prompt_template.format(context=context, question=question)

In [12]:
data = llm(prompt=prompt)
print(data)

Yes — you can join the course, as it starts on 8th June 2026.


In [13]:
## Now lets import some docs into memory

In [14]:
import requests
from tqdm.auto import tqdm

In [15]:
courses_url = "https://datatalks.club/faq/json/courses.json"
courses_faq_base_url = "https://datatalks.club/faq"

In [16]:
response = requests.get(courses_url)
courses = response.json()

documents = []
for course in tqdm(courses, desc="Processing courses"):
    course_faq_url = f"{courses_faq_base_url}{course['path']}"
    faq_response = requests.get(course_faq_url)
    faq_response.raise_for_status()
    faq_data = faq_response.json()
    documents.extend(faq_data)

print(len(documents))

Processing courses:   0%|          | 0/6 [00:00<?, ?it/s]

1346


In [17]:
from minsearch import Index

In [18]:
index = Index(
    text_fields = ["question", "answer", "section"],
    keyword_fields = ["course"]
)
index.fit(documents)

In [19]:
index.search(query=question,
             boost_dict={"question": 2.0, "section" : 0.5},
             filter_dict={"course": "data-engineering-zoomcamp"},
             num_results=5)

[{'id': '3f1424af17',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."},
 {'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slac

In [20]:
def build_context(results: list) -> str:
    docs = []
    for result in results:
        docs.append(f"Section: {result['section']}")
        docs.append(f"Question: {result['question']}")
        docs.append(f"Answer: {result['answer']}")
        docs.append(" ")
    return "\n".join(docs).strip()

def search_results(question: str) -> list:
    results = index.search(query=question,
             boost_dict={"question": 2.0, "section" : 0.5},
             filter_dict={"course": "data-engineering-zoomcamp"},
             num_results=5)

    return results

In [21]:
results = search_results(question)
print(build_context(results))

Section: General Course-Related Questions
Question: Course: Can I still join the course after the start date?
Answer: Yes, even if you don't register, you're still eligible to submit the homework.

Be aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.
 
Section: General Course-Related Questions
Question: Course: When does the course start?
Answer: A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).

- Register via the link in the course repo before the cohort starts.
- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.
- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.
 
Section: General Course-Related Questions
Question: Course - Can I follow the 

In [22]:
prompt_template = """You are a course FAQ assistant, your role is to answer student queries
You will be asked a question, and you should answer it based on the following context, which contains the 
most relevant FAQ entries from the course:

{context}

Question to answer: {question}
"""

In [23]:
def llm_call(prompt: str) -> str:
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=[{"role": "user", "content": prompt}]
    )
    return response.output_text

In [24]:
def llm(question: str) -> str:
    results = search_results(question)
    context = build_context(results)
    prompt = prompt_template.format(context=context, question=question)
    response = llm_call(prompt)
    return response

In [25]:
print(llm(question))

Yes — you can still join, even if the course has already started. You’re still eligible to submit the homework.

Just keep in mind that homework and final project deadlines still apply, so it’s best not to leave everything until the last minute. If the current cohort has already finished, you can also follow the course at your own pace since all materials remain available.


In [28]:
class FaqRag:
    def __init__(self, index: Index, prompt_template: str, system_instruction: str = "", model_name: str = "gpt-5.4-mini"):
        self.index = index
        self.prompt_template = prompt_template
        self.system_instruction = system_instruction
        self.model_name = "gpt-5.4-mini" if not model_name else model_name
        self._client = OpenAI()

    @staticmethod
    def build_context(results: list) -> str:
        docs = []
        for result in results:
            docs.append(f"Section: {result['section']}")
            docs.append(f"Question: {result['question']}")
            docs.append(f"Answer: {result['answer']}")
            docs.append(" ")
        return "\n".join(docs).strip()

    def search_results(self, question: str, filters: dict, num_results: int=5) -> list:
        results = self.index.search(query=question,
                boost_dict={"question": 2.0, "section" : 0.5},
                filter_dict=filters,
                num_results=num_results)

        return results
    
    def llm_call(self, prompt: str) -> str:
        """Make a call to the LLM and return the response text."""

        history = [
            {"role": "system", "content": self.system_instruction},
            {"role": "user", "content": prompt}
        ]
        
        response = self._client.responses.create(
            model=self.model_name,
            input=history
        )
        return response.output_text
    
    def rag(self, question: str, filters: dict, num_results: int=5) -> str:
        results = self.search_results(question, filters, num_results)
        context = build_context(results)
        prompt = self.prompt_template.format(context=context, question=question)
        response = self.llm_call(prompt)
        return response

In [30]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = """
<USER QUESTION>
{question}
</USER QUESTION>

<CONTEXT>
{context}
</CONTEXT>
"""

In [32]:
rag = FaqRag(
    index=index,
    prompt_template=PROMPT_TEMPLATE,
    system_instruction=INSTRUCTIONS
)

In [33]:
user_query_response = rag.rag(
    question="I just discovered the course. Can I join?",
    filters={"course": "data-engineering-zoomcamp"},
    num_results=5)

print(user_query_response)

Yes, you can still join. Even if you missed the start or don’t register, you’re still eligible to submit the homework.

Just keep in mind that there are deadlines for homework and the final projects, so don’t leave everything until the last minute.
